## **Inference Model**

In [13]:
import pandas as pd
from bert_score import score

# Load data
results_df = pd.read_csv('inference_results.csv')
truth_df = pd.read_csv('dataset_clean_fixed.csv')

# Merge to ensure alignment
df = results_df.merge(truth_df, on = "id")

# Drop missing values
df = df.dropna(subset = ["answer", "correct_answer"])

# Prepare lists
df['answer_short'] = df['answer'].str[:300]
cands = df['answer_short'].tolist()
refs = df['correct_answer'].tolist()

# Compute BERTScore
P, R, F1 = score(
    cands,
    refs,
    lang = "de",
    rescale_with_baseline = True,
    verbose = True
)

# Print results
print(f"BERTScore Precision: {P.mean().item():.4f}")
print(f"BERTScore Recall: {R.mean().item():.4f}")
print(f"Final Mean BERTScore F1: {F1.mean().item():.4f}")

# Save results
df['bert_f1'] = F1.tolist()
df.to_csv('final_evaluations.csv', index = False)

"""Results:
BERTScore Precision: 0.1768
BERTScore Recall: 0.2163
Final Mean BERTScore F1: 0.1956
"""

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


calculating scores...
computing bert embedding.


  0%|          | 0/18 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/9 [00:00<?, ?it/s]

done in 25.28 seconds, 22.20 sentences/sec
BERTScore Precision: 0.1768
BERTScore Recall: 0.2163
Final Mean BERTScore F1: 0.1956


## **Fine-tune Model**

In [14]:
from bert_score import score
import pandas as pd

# Load fine-tuned results and the reference dataset
results_df = pd.read_csv('fine_tuned_results.csv')
reference_df = pd.read_csv('dataset_clean_fixed.csv') # Assuming you have ground truth

# Merge to ensure alignment
df = results_df.merge(truth_df, on = "id")

# Drop missing values
df = df.dropna(subset = ["answer", "correct_answer"])

candidates = df['answer'].tolist()
references = df['correct_answer'].tolist()

# Calculate BERTScore
# lang = "de" is used for Austrian/German Tax Law context
P, R, F1 = score(candidates, references, lang = "de", verbose = True)

print(f"BERTScore Precision: {P.mean():.4f}")
print(f"BERTScore Recall: {R.mean():.4f}")
print(f"BERTScore F1: {F1.mean():.4f}")

# Add scores back to dataframe for analysis
results_df['bert_f1'] = F1.tolist()
results_df.to_csv('evaluated_results.csv', index = False)

"""
Results:
BERTScore Precision: 0.7450
BERTScore Recall: 0.7129
BERTScore F1: 0.7277
"""

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


calculating scores...
computing bert embedding.


  0%|          | 0/18 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/9 [00:00<?, ?it/s]

done in 22.70 seconds, 24.72 sentences/sec
BERTScore Precision: 0.7450
BERTScore Recall: 0.7129
BERTScore F1: 0.7277


ValueError: Length of values (561) does not match length of index (643)

# **RAG Model**

In [10]:
import pandas as pd
from evaluate import load

def run_evaluation():
    """
    Evaluates the RAG output using BERTScore (Precision, Recall, F1)
    by comparing the generated 'answer' against the 'correct_answer'.
    """
    # Load the generated results
    results_df = pd.read_csv("rag_results.csv")

    # Load the test dataset (dataset_clean.csv)
    test_df = pd.read_csv("dataset_clean_fixed.csv")

    # Merge on 'id' to pair answers with their original prompts
    eval_df = results_df.merge(test_df, on = 'id')

    # Initialize the metric
    bertscore = load("bertscore")

    # Use 'answer' as prediction and 'correct_answer' as the reference
    predictions = eval_df['answer'].fillna("").tolist()
    references = eval_df['correct_answer'].tolist()

    print(f"Evaluating {len(predictions)} samples...")

    # BERTScore calculation for German language
    bs_results = bertscore.compute(predictions = predictions, references = references, lang = "de")

    # Calculate averages
    avg_precision = sum(bs_results['precision']) / len(bs_results['precision'])
    avg_recall = sum(bs_results['recall']) / len(bs_results['recall'])
    avg_f1 = sum(bs_results['f1']) / len(bs_results['f1'])

    # Output Final Report
    print("\nBERTScore Evaluation Metrics")
    print(f"Average Precision: {avg_precision:.4f}")
    print(f"Average Recall: {avg_recall:.4f}")
    print(f"Average F1 Score: {avg_f1:.4f}")

# Execute
run_evaluation()

"""
Results:
Average Precision: 0.6986
Average Recall:    0.6815
Average F1 Score:  0.6878
"""

Evaluating 643 samples...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



BERTScore Evaluation Metrics (Answer vs. Prompt)
Average Precision: 0.6986
Average Recall:    0.6815
Average F1 Score:  0.6878
